# 🩺 Mammogram 3-Class Classification with Advanced XAI
**Target:** Normal (BI-RADS 1), Benign (BI-RADS 3), Malignant (BI-RADS 4/5)  
**Dataset:** King Abdulaziz University Mammogram Dataset  
**Key Methodological Rigor:** Patient-level splitting (zero data leakage), CLAHE enhancement, Grad-CAM++ for microcalcification localization.

In [ ]:
# 1. Imports & Environment Setup
import os
import re
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import tensorflow as tf
from tensorflow.keras import layers, models, applications
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 2. Robust Data Loading & Patient ID Extraction
We map the specific Kaggle nested directories and extract the unique Patient ID (`BC\d+`) from the filenames to ensure strict patient-level splitting later.

In [ ]:
BASE_DIR = '/kaggle/input/datasets/asmaasaad/king-abdulaziz-university-mammogram-dataset'

# Exact nested subdirectories as provided in the dataset
folder_mapping = {
    'BIRAD1/b1': 0,        # Normal
    'Birad3/b3': 1,        # Benign
    'Birad4/b4': 2,        # Malignant
    'Birad5/Birad5': 2     # Malignant
}

class_names = ['Normal (BI-RADS 1)', 'Benign (BI-RADS 3)', 'Malignant (BI-RADS 4/5)']
patient_id_pattern = re.compile(r'(BC\d+)')

image_paths, labels, patient_ids = [], [], []

for rel_path, label in folder_mapping.items():
    full_path = os.path.join(BASE_DIR, rel_path)
    
    # Fault tolerance: fallback to parent folder if exact nested path fails
    if not os.path.exists(full_path):
        parent_path = os.path.join(BASE_DIR, rel_path.split('/')[0])
        if os.path.exists(parent_path):
            print(f"Warning: '{full_path}' not found. Falling back to: '{parent_path}'")
            full_path = parent_path
        else:
            print(f"Error: '{full_path}' missing. Skipping.")
            continue
            
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.tiff', '*.bmp']:
        for filepath in glob.glob(os.path.join(full_path, '**', ext), recursive=True):
            image_paths.append(filepath)
            labels.append(label)
            
            filename = os.path.basename(filepath)
            match = patient_id_pattern.search(filename)
            patient_ids.append(match.group(1) if match else filename)

print(f"Total images loaded: {len(image_paths)}")
print(f"Unique patient IDs found: {len(set(patient_ids))}")

# Sanity Check
print("\n--- Sanity Check (First 3 Images) ---")
for i in range(min(3, len(image_paths))):
    print(f"Patient ID: {patient_ids[i]} | Label: {class_names[labels[i]]}")

## 3. Patient-Level Data Splitting (Zero Leakage)
Using `GroupShuffleSplit` to ensure that NO patient's CC/MLO or Left/Right views cross between Train, Validation, and Test sets.

In [ ]:
# 1st Split: Train (70%) vs Temp (30%)
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, temp_idx = next(gss1.split(image_paths, labels, groups=patient_ids))

X_train = [image_paths[i] for i in train_idx]
y_train = [labels[i] for i in train_idx]

paths_temp = [image_paths[i] for i in temp_idx]
labels_temp = [labels[i] for i in temp_idx]
groups_temp = [patient_ids[i] for i in temp_idx]

# 2nd Split: Temp into Val (50% of 30% = 15%) and Test (15%)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(gss2.split(paths_temp, labels_temp, groups=groups_temp))

X_val = [paths_temp[i] for i in val_idx]
y_val = [labels_temp[i] for i in val_idx]
X_test = [paths_temp[i] for i in test_idx]
y_test = [labels_temp[i] for i in test_idx]

print(f"Patient-Level Split:")
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

# Verify ZERO patient overlap between Train and Test
train_pts = set([patient_id_pattern.search(os.path.basename(p)).group(1) for p in X_train if patient_id_pattern.search(os.path.basename(p))])
test_pts = set([patient_id_pattern.search(os.path.basename(p)).group(1) for p in X_test if patient_id_pattern.search(os.path.basename(p))])
overlap = train_pts.intersection(test_pts)
print(f"Patient Overlap (Train vs Test): {len(overlap)} (MUST BE 0)")

## 4. Preprocessing (CLAHE) & Memory-Efficient Data Generator

In [ ]:
def preprocess_mammogram(image_path, target_size=(512, 512)):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img_clahe = clahe.apply(img)
    img_resized = cv2.resize(img_clahe, target_size)
    img_normalized = img_resized.astype(np.float32) / 255.0
    return np.stack((img_normalized,) * 3, axis=-1)

def data_generator(paths, labels, batch_size=16, target_size=(512, 512), augment=False):
    while True:
        for i in range(0, len(paths), batch_size):
            batch_paths = paths[i:i+batch_size]
            batch_labels = labels[i:i+batch_size]
            images, valid_labels = [], []
            
            for path, label in zip(batch_paths, batch_labels):
                img = preprocess_mammogram(path, target_size)
                if img is not None:
                    if augment:
                        img = tf.image.random_flip_left_right(img)
                        img = tf.image.random_flip_up_down(img)
                    images.append(img)
                    valid_labels.append(label)
            
            if len(images) > 0:
                yield np.array(images), tf.keras.utils.to_categorical(valid_labels, num_classes=3)

## 5. Model Architecture (Transfer Learning with EfficientNetB0)

In [ ]:
IMG_SIZE = (512, 512)
BATCH_SIZE = 16

base_model = applications.EfficientNetB0(
    include_top=False, weights='imagenet', input_shape=(*IMG_SIZE, 3), pooling='avg'
)
base_model.trainable = False

inputs = layers.Input(shape=(*IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(3, activation='softmax')(x)

model = models.Model(inputs, outputs)

# Calculate class weights to handle BI-RADS imbalance
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))
print(f"Class Weights: {class_weight_dict}")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(multi_class='ovr', name='auc')]
)
model.summary()

## 6. Model Training

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('best_mammogram_model.h5', monitor='val_auc', save_best_only=True, mode='max'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)
]

train_gen = data_generator(X_train, y_train, batch_size=BATCH_SIZE, augment=True)
val_gen = data_generator(X_val, y_val, batch_size=BATCH_SIZE, augment=False)

history = model.fit(
    train_gen, steps_per_epoch=len(X_train) // BATCH_SIZE,
    validation_data=val_gen, validation_steps=len(X_val) // BATCH_SIZE,
    epochs=30, class_weight=class_weight_dict, callbacks=callbacks
)

## 7. Model Evaluation

In [ ]:
test_gen = data_generator(X_test, y_test, batch_size=BATCH_SIZE, augment=False)
y_pred_probs = model.predict(test_gen, steps=len(X_test) // BATCH_SIZE + 1)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_true_classes = np.array(y_test[:len(y_pred_classes)])

print("\nClassification Report:")
print(classification_report(y_true_classes, y_pred_classes, target_names=class_names))

cm = confusion_matrix(y_true_classes, y_pred_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix (Patient-Level Split)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## 8. Advanced XAI: Grad-CAM vs. Grad-CAM++
Grad-CAM++ utilizes higher-order derivatives to better localize sparse, fine-grained features like microcalcifications compared to standard Grad-CAM.

In [ ]:
def get_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model([model.inputs], [model.get_layer(last_conv_layer_name).output, model.output])
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None: pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = last_conv_layer_output[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    return tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-7), pred_index.numpy()

def get_gradcam_plus_plus(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model([model.inputs], [model.get_layer(last_conv_layer_name).output, model.output])
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(model.inputs)
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None: pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
        grads1 = tape.gradient(class_channel, last_conv_layer_output)
        grads2 = tape.gradient(grads1, last_conv_layer_output)
        grads3 = tape.gradient(grads2, last_conv_layer_output)
        
    grads1_sq = tf.square(grads1)
    grads1_cu = grads1_sq * grads1
    denom = 2 * grads1_sq + tf.reduce_sum(grads1_cu * last_conv_layer_output, axis=(1, 2), keepdims=True)
    alpha = tf.nn.relu(grads1_sq / (denom + 1e-7))
    weights = tf.reduce_sum(alpha * tf.nn.relu(grads1), axis=(1, 2))
    
    cam = tf.reduce_sum(weights[..., tf.newaxis, tf.newaxis] * tf.nn.relu(last_conv_layer_output), axis=-1)
    cam = tf.squeeze(cam)
    return tf.maximum(cam, 0) / (tf.math.reduce_max(cam) + 1e-7), pred_index.numpy()

def display_xai_comparison(img_path, true_label, model, last_conv_layer_name='top_conv'):
    img = preprocess_mammogram(img_path, (512, 512))
    if img is None: return
    img_array = np.expand_dims(img, axis=0)
    preds = model.predict(img_array, verbose=0)
    pred_class, conf = np.argmax(preds[0]), preds[0][np.argmax(preds[0])]
    
    heatmap_cam, _ = get_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_class)
    heatmap_campp, _ = get_gradcam_plus_plus(img_array, model, last_conv_layer_name, pred_class)
    
    jet = plt.colormaps['jet']
    cam_colored = jet(heatmap_cam)[..., :3] * 255
    campp_colored = jet(heatmap_campp)[..., :3] * 255
    
    orig = cv2.resize(cv2.imread(img_path, cv2.IMREAD_GRAYSCALE), (512, 512))
    orig_3ch = np.stack((orig,) * 3, axis=-1)
    super_campp = np.clip(campp_colored * 0.4 + orig_3ch * 0.6, 0, 255).astype(np.uint8)
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(orig, cmap='gray'); axes[0].set_title(f'Original (CLAHE)\nTrue: {class_names[true_label]}'); axes[0].axis('off')
    axes[1].imshow(heatmap_cam, cmap='jet'); axes[1].set_title('Standard Grad-CAM'); axes[1].axis('off')
    axes[2].imshow(heatmap_campp, cmap='jet'); axes[2].set_title('Grad-CAM++ (Recommended)'); axes[2].axis('off')
    axes[3].imshow(super_campp); axes[3].set_title(f'Prediction: {class_names[pred_class]}\nConf: {conf:.2f}'); axes[3].axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
# Run XAI on 3 random test samples
import random
sample_indices = random.sample(range(len(X_test)), 3)
for i in sample_indices:
    display_xai_comparison(X_test[i], y_test[i], model, last_conv_layer_name='top_conv')

---
## 📝 Critical Evaluation for Publication (Read Before Submitting)

**1. Evaluation of Your Proposed Work:**
* **Strengths:** Targeting 3-class (Normal/Benign/Malignant) is clinically highly relevant. Using the KAU dataset provides a unique demographic contribution. 
* **Critical Flaw Fixed:** Your initial plan lacked patient-level splitting. By extracting the `BC\d+` ID and using `GroupShuffleSplit`, we eliminated the #1 cause of medical AI paper rejections (data leakage via multiple views of the same patient).

**2. Evaluation of the Provided Pipeline:**
* **Strengths:** Implements CLAHE (matching the PMC5274695 baseline), handles class imbalance via `class_weight`, and utilizes Grad-CAM++ which is mathematically superior for localizing sparse microcalcifications compared to standard Grad-CAM.
* **Limitations to Disclose in Your Paper:** 
  1. *Resolution Trade-off:* Resizing to 512x512 saves VRAM but may dilute microcalcifications. Mention "Patch-based Multiple Instance Learning (MIL)" as future work.
  2. *External Validation:* This model is trained and tested only on KAU data. To get into a Q1 journal, you **must** test this exact saved model (`best_mammogram_model.h5`) on an external dataset like CBIS-DDSM to prove it hasn't overfit to the KAU scanner's specific noise profile.
  3. *Clinical Fidelity of XAI:* Grad-CAM++ shows *where* the model looks, but a radiologist must confirm it's looking at the *right* thing. Include a section where a certified radiologist scores 50 random Grad-CAM++ outputs for clinical relevance.